In [1]:
# =============================================================================
# GUS02D: Additional Data
# =============================================================================
# 
# ...
#
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc
import random

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'
json_root = gus_root / 'data' / 'extracted'

print(f"Repository root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"JSON root: {json_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS
JSON root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS/data/extracted


In [2]:
# We load jsons from the API extraction and inspect them here. We load first only the files that start with "pop_" and are .jsons.
json_filenames = list(json_root.glob("pop_*.json"))
json_files = {}
for filename in json_filenames:
    with open(filename, "r") as f:
        json_files[filename.stem] = f.read()

In [3]:

# =============================================================================
# GUS02D PART 1: Extract raw age-distribution tables from all pop_age JSON files
# =============================================================================
#
# For every JSON in json_files (both pop_age_YYYY_f and pop_age_men_YYYY_f):
#   - Extract a DataFrame:  rows = territorial units, cols = canonical age groups
#   - Row index uses name_pl from PRE_1999_VOIVODESHIPS (or 'Polska')
#   - Canonical column names: 'ogółem', '0', '1-4', ..., '65-69', '70 i więcej'
#   - Left and right table halves are joined ONLY when Lp. sets match exactly
#   - Validation: sum of age-group columns must equal 'ogółem' for each row
#   - Files with validation issues → saved as <stem>_validate.xlsx for manual review
# =============================================================================

import json, re, unicodedata, difflib, pickle
from io import StringIO
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Canonical age column order
# ---------------------------------------------------------------------------
_LEFT_AGE_COLS  = ['ogółem', '0', '1-4', '5-9', '10-14', '15-19', '20-24']
_RIGHT_AGE_COLS = ['25-29', '30-34', '35-39', '40-44', '45-49',
                   '50-54', '55-59', '60-64', '65-69', '70 i więcej']
_CANONICAL_AGE_COLS = _LEFT_AGE_COLS + _RIGHT_AGE_COLS

# ---------------------------------------------------------------------------
# Name normalisation and matching → returns name_pl
# ---------------------------------------------------------------------------

def _norm(s: str) -> str:
    s = str(s).lower()
    for f, t in [('ł','l'),('ą','a'),('ę','e'),('ó','o'),('ś','s'),
                 ('ź','z'),('ż','z'),('ć','c'),('ń','n')]:
        s = s.replace(f, t)
    s = ''.join(c for c in unicodedata.normalize('NFD', s)
                if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', s)).strip()

# Build lookup: normalized(name_pl) → name_pl
_norm_to_name_pl: dict[str, str] = {
    _norm(v['name_pl']): v['name_pl']
    for v in gtdb.PRE_1999_VOIVODESHIPS.values()
}

def _match_name_pl(ocr_name: str) -> str | None:
    """Return canonical name_pl or 'Polska'; None if no match."""
    raw = ocr_name.strip()
    # Stripped of spaces, replacing OCR '0'→'O'
    ns = re.sub(r'\s+', '', raw.upper()).replace('0', 'O')
    if ns in ('POLSKA', 'POLAND', 'POLSKAPOLAND', 'OGÓŁEM', 'OGOLEM',
              'PLOAND', 'POLLAND'):
        return 'Polska'

    n = _norm(raw)

    # 1. Exact
    if n in _norm_to_name_pl:
        return _norm_to_name_pl[n]

    # 2. Substring containment (canonical name inside OCR text)
    for cn, cp in _norm_to_name_pl.items():
        if cn in n:
            return cp

    # 3. Fuzzy
    best_r, best_cp = 0.0, None
    for cn, cp in _norm_to_name_pl.items():
        r = difflib.SequenceMatcher(None, n, cn).ratio()
        if r > best_r:
            best_r, best_cp = r, cp
    if best_r >= 0.70:
        return best_cp

    # 4. Keyword fallback for POLSKA
    if re.search(r'\bpolsk|\bpolan', n):
        return 'Polska'

    return None

_ALL_EXPECTED = {'Polska'} | {v['name_pl'] for v in gtdb.PRE_1999_VOIVODESHIPS.values()}

# ---------------------------------------------------------------------------
# Column normalisation → canonical age label
# ---------------------------------------------------------------------------

def _match_age_col(raw: str) -> str | None:
    c = str(raw).strip()
    c = re.sub(r'-{2,}', '-', c)
    c = re.sub(r'(\d)\s*-\s*(\d)', r'\1-\2', c)
    c = re.sub(r'(\d)\s+(\d{1,2})\b', r'\1-\2', c)
    cl = c.lower()

    # Total
    if re.search(r'og[oó][łl]|^total$|^razem$|^ogtem$|^oglem$', cl):
        return 'ogółem'
    # 70+
    if re.search(r'70\s*(lat|i\s+wi|years|jat|vears)', cl, re.I) or re.match(r'^70[+]?$', c):
        return '70 i więcej'
    # 0 infants
    if re.match(r'^0(\s+lat|\s+years|$)', cl):
        return '0'
    # Numeric range x-y
    m = re.match(r'^(\d+)-(\d+)$', c)
    if m:
        key = f"{int(m.group(1))}-{int(m.group(2))}"
        if key in _CANONICAL_AGE_COLS:
            return key
    return None

# ---------------------------------------------------------------------------
# General table-parsing utilities
# ---------------------------------------------------------------------------

def _flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    new = []
    for col in df.columns:
        if isinstance(col, tuple):
            l1, l0 = str(col[1]).strip(), str(col[0]).strip()
            new.append(l1 if (l1 and not l1.startswith('Unnamed') and l1 != 'nan') else l0)
        else:
            new.append(str(col).strip())
    df.columns = new
    return df

def _clean_num(val) -> float:
    if isinstance(val, (int, float)):
        return float(val) if not pd.isna(val) else float('nan')
    s = re.sub(r'[^\d]', '', str(val))
    return float(s) if s else float('nan')

def _clean_lp(val) -> int | None:
    try:
        f = float(str(val).strip())
        i = int(round(f))
        if abs(f - i) < 0.01 and 1 <= i <= 50:
            return i
    except (ValueError, TypeError):
        pass
    digits = re.sub(r'[^\d]', '', str(val))
    if digits and 1 <= len(digits) <= 2:
        v = int(digits)
        return v if 1 <= v <= 50 else None
    return None

def _find_lp_col(df: pd.DataFrame, prefer_last: bool = False) -> str:
    for c in df.columns:
        if re.search(r'[Ll][Pp]\b|[Ii]\s*\.\s*[Pp]', str(c)):
            return c
    return df.columns[-1] if prefer_last else df.columns[0]

def _all_cols_numeric(df: pd.DataFrame) -> bool:
    return all(re.match(r'^\d+(\.\d+)?$', str(c).strip()) for c in df.columns)

# ---------------------------------------------------------------------------
# Parse one table half (left or right side of the scanned page)
# ---------------------------------------------------------------------------

def _parse_half(html: str, is_right: bool, pos_labels: list[str]) -> pd.DataFrame | None:
    """
    Returns DataFrame with: _lp (int), _name (str, left only), + resolved age cols.
    Returns None on failure.
    """
    try:
        raw = _flatten_cols(pd.read_html(StringIO(html))[0])
    except Exception as e:
        print(f"      [ERROR] read_html: {e}")
        return None

    # If POLSKA row was consumed as column header (all-numeric headers)
    if _all_cols_numeric(raw):
        saved = list(raw.columns)
        tcols = [f'_p{i}' for i in range(len(saved))]
        raw.columns = tcols
        raw = pd.concat([pd.DataFrame([saved], columns=tcols), raw], ignore_index=True)

    lp_col  = _find_lp_col(raw, prefer_last=is_right)
    lp_vals = raw[lp_col].apply(_clean_lp)
    valid   = lp_vals.notna()

    df = raw[valid].copy().reset_index(drop=True)
    df['_lp'] = lp_vals[valid].astype(int).reset_index(drop=True)
    df = df.drop_duplicates(subset=['_lp'], keep='first')
    df = df.sort_values('_lp').reset_index(drop=True)

    non_lp = [c for c in df.columns if c not in (lp_col, '_lp')]
    if is_right:
        data_cols = non_lp
    else:
        # First non-lp column is the name column (voivodeship names)
        data_cols = non_lp[1:] if len(non_lp) > 1 else []

    # Map column names → canonical age labels; fall back to positional
    resolved: dict[str, list] = {}
    unresolved_pos: dict[int, list] = {}
    for pos, col in enumerate(data_cols):
        label = _match_age_col(col)
        vals  = df[col].apply(_clean_num).tolist()
        if label:
            if label not in resolved:
                resolved[label] = vals
        else:
            unresolved_pos[pos] = vals

    for pos, vals in unresolved_pos.items():
        if pos < len(pos_labels) and pos_labels[pos] not in resolved:
            resolved[pos_labels[pos]] = vals

    result = pd.DataFrame({'_lp': df['_lp'].values})
    if not is_right:
        name_col = non_lp[0] if non_lp else None
        result['_name'] = df[name_col].astype(str).values if name_col else ''
    for label, vals in resolved.items():
        if len(vals) == len(result):
            result[label] = vals

    return result.sort_values('_lp').reset_index(drop=True)

# ---------------------------------------------------------------------------
# Extract full table from one JSON (joins two halves iff Lp. sets match)
# ---------------------------------------------------------------------------

def _extract_table(data: dict, stem: str) -> pd.DataFrame | None:
    tables = [e for e in data['elements'] if e['category'] == 'table']
    if len(tables) < 2:
        print(f"  [WARN {stem}] Only {len(tables)} table element(s), expected 2")

    left  = _parse_half(tables[0]['content']['html'], is_right=False,
                        pos_labels=_LEFT_AGE_COLS) if tables else None
    right = _parse_half(tables[1]['content']['html'], is_right=True,
                        pos_labels=_RIGHT_AGE_COLS) if len(tables) >= 2 else None

    if left is None and right is None:
        print(f"  [ERROR {stem}] Both halves failed to parse")
        return None

    # Join halves ONLY if Lp. sets match exactly (100% sure alignment)
    if left is not None and right is not None:
        lp_l, lp_r = set(left['_lp']), set(right['_lp'])
        if lp_l == lp_r:
            right_data = right[[c for c in right.columns if c != '_name']]
            combined = left.merge(right_data, on='_lp', how='inner')
        else:
            only_l = sorted(lp_l - lp_r)
            only_r = sorted(lp_r - lp_l)
            print(f"  [WARN {stem}] Lp. mismatch → only left half used. "
                  f"Only-in-left: {only_l[:5]}, Only-in-right: {only_r[:5]}")
            combined = left
    elif left is not None:
        combined = left
    else:
        combined = right  # last resort if left failed entirely

    # Resolve row names → canonical name_pl
    rows, unmatched = [], []
    for _, row in combined.iterrows():
        raw_name = str(row.get('_name', '')) if '_name' in combined.columns else ''
        name_pl  = _match_name_pl(raw_name)
        if name_pl is None:
            unmatched.append(raw_name)
            continue
        vals = {col: (row[col] if col in combined.columns else float('nan'))
                for col in _CANONICAL_AGE_COLS if col in combined.columns}
        vals['_row'] = name_pl
        rows.append(vals)

    if unmatched:
        print(f"  [WARN {stem}] Unmatched row names: {unmatched}")
    if not rows:
        print(f"  [ERROR {stem}] No rows resolved")
        return None

    df_out = pd.DataFrame(rows).set_index('_row')
    df_out.index.name = None
    # Reorder to canonical column order
    df_out = df_out[[c for c in _CANONICAL_AGE_COLS if c in df_out.columns]].astype(float)
    return df_out

# ---------------------------------------------------------------------------
# Validation: row sum of age groups == ogółem
# ---------------------------------------------------------------------------

def _validate(df: pd.DataFrame, stem: str, tol: float = 2.0) -> list[str]:
    warnings = []
    age_cols = [c for c in df.columns if c != 'ogółem']
    if 'ogółem' not in df.columns or not age_cols:
        return warnings
    row_sums = df[age_cols].sum(axis=1)
    diff = (row_sums - df['ogółem']).abs()
    for idx in diff.index:
        if not pd.isna(diff[idx]) and diff[idx] > tol:
            warnings.append(
                f"  '{idx}': Σage_groups={row_sums[idx]:.0f} ≠ ogółem={df.at[idx,'ogółem']:.0f} "
                f"(diff={diff[idx]:.0f})")
    return warnings

# ---------------------------------------------------------------------------
# Main extraction loop
# ---------------------------------------------------------------------------

save_dir = gus_root / 'data' / 'extracted_tables'
save_dir.mkdir(parents=True, exist_ok=True)

extracted_tables: dict[str, pd.DataFrame] = {}   # stem → df
validation_issues: dict[str, list[str]]   = {}   # stem → warnings

for stem in sorted(json_files):
    data = json.loads(json_files[stem])
    print(f"\n{stem}")

    df = _extract_table(data, stem)
    if df is None:
        print("  → SKIP (extraction failed)")
        continue

    extracted_tables[stem] = df

    # Coverage check
    missing = _ALL_EXPECTED - set(df.index)
    if missing:
        print(f"  Missing {len(missing)} unit(s): {sorted(missing)}")

    # Validate sums
    warns = _validate(df, stem)
    if warns:
        validation_issues[stem] = warns
        print(f"  ✗ Validation issues ({len(warns)} rows):")
        for w in warns:
            print(w)

    # Save xlsx if there are sum errors OR missing/unmatched units
    needs_review = bool(warns) or bool(missing)
    if needs_review:
        if stem not in validation_issues:
            validation_issues[stem] = []
        xlsx_path = save_dir / f"{stem}_validate.xlsx"
        df.to_excel(xlsx_path)
        reasons = []
        if warns:
            reasons.append(f"{len(warns)} sum mismatch(es)")
        if missing:
            reasons.append(f"{len(missing)} missing unit(s): {sorted(missing)}")
        print(f"  → Saved for review ({', '.join(reasons)}): {xlsx_path.name}")
    else:
        print(f"  ✓ {len(df)} units — all column sums OK, no missing units")

print(f"\n{'='*60}")
print(f"Extracted:        {len(extracted_tables)}/{len(json_files)} files")
print(f"Need review:      {len(validation_issues)} file(s)")
if validation_issues:
    for s in sorted(validation_issues):
        print(f"  {s}")
print(f"Review folder:    {save_dir}")



pop__tot_age_men_educ_1986 [meta]
  [WARN pop__tot_age_men_educ_1986 [meta]] Only 0 table element(s), expected 2
  [ERROR pop__tot_age_men_educ_1986 [meta]] Both halves failed to parse
  → SKIP (extraction failed)

pop__tot_age_men_educ_1987 [meta]
  [WARN pop__tot_age_men_educ_1987 [meta]] Only 1 table element(s), expected 2
  [ERROR pop__tot_age_men_educ_1987 [meta]] No rows resolved
  → SKIP (extraction failed)

pop__tot_age_men_educ_1988 [meta]
  [WARN pop__tot_age_men_educ_1988 [meta]] Only 1 table element(s), expected 2
  [ERROR pop__tot_age_men_educ_1988 [meta]] No rows resolved
  → SKIP (extraction failed)

pop__tot_age_men_educ_1991 [meta]
  [WARN pop__tot_age_men_educ_1991 [meta]] Only 1 table element(s), expected 2
  [ERROR pop__tot_age_men_educ_1991 [meta]] No rows resolved
  → SKIP (extraction failed)

pop__tot_age_men_educ_1992 [meta]
  [WARN pop__tot_age_men_educ_1992 [meta]] Only 1 table element(s), expected 2
  [ERROR pop__tot_age_men_educ_1992 [meta]] No rows resolve

In [4]:

# =============================================================================
# GUS02D PART 2: Build age × gender cross-tables and save as pickle
# =============================================================================
#
# For each year 1986–1994 and each territorial unit (49 voivodeships + Polska):
#   - Load the age-distribution table for total population  (pop_age_YYYY_f)
#   - Load the age-distribution table for male  population  (pop_age_men_YYYY_f)
#   - Source priority: corrected xlsx in save_dir (after manual review) → re-extract from JSON
#   - Build cross-table DataFrame:  rows = age groups, cols = ['ogółem', 'mężczyźni', 'kobiety']
#     where kobiety = ogółem − mężczyźni
#   - Collect result in:  cross_tables[unit_name_pl][year_str] = pd.DataFrame
#   - Save the full dict as pickle at gus_root / pop_age_cross_tables.pkl
# =============================================================================

import pickle

YEARS = list(range(1986, 1995))

# Column names for the cross-table
_CT_TOTAL  = 'ogółem'
_CT_MALE   = 'mężczyźni'
_CT_FEMALE = 'kobiety'

# ---------------------------------------------------------------------------
# Loader: xlsx from review folder (corrected) or re-extract from JSON
# ---------------------------------------------------------------------------

def _load_table(stem: str) -> pd.DataFrame | None:
    """
    Load an extracted age-distribution table for `stem`.
    Precedence:
      1. <save_dir>/<stem>.xlsx  (manually corrected — user removed '_validate')
      2. Re-extract from json_files[stem] using functions defined in Part 1
    """
    xlsx_path = save_dir / f"{stem}.xlsx"
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, index_col=0)
        df.columns = [str(c) for c in df.columns]
        print(f"    Loaded from xlsx: {xlsx_path.name}")
        return df.astype(float)

    if stem in json_files:
        data = json.loads(json_files[stem])
        df = _extract_table(data, stem)
        if df is not None:
            return df
        print(f"    [WARN] Re-extraction failed for {stem}")
        return None

    print(f"    [WARN] No source found for {stem}")
    return None

# ---------------------------------------------------------------------------
# Validation for cross-tables
# ---------------------------------------------------------------------------

def _validate_cross_table(df: pd.DataFrame, unit: str, year: int, tol: float = 2.0):
    """
    Two checks:
      1. Sum of age-group rows (excl. total) == total row  (column-wise)
      2. mężczyźni + kobiety == ogółem  (row-wise)
    """
    age_rows = [r for r in df.index if r != 'ogółem']

    # Check 1: column sums
    if 'ogółem' in df.index and age_rows:
        col_sums = df.loc[age_rows].sum(axis=0)
        col_tot  = df.loc['ogółem']
        diff = (col_sums - col_tot).abs()
        for col in diff.index:
            if not pd.isna(diff[col]) and diff[col] > tol:
                print(f"    [VALID col-sum {unit} {year}] '{col}': "
                      f"Σrows={col_sums[col]:.0f} ≠ total={col_tot[col]:.0f} "
                      f"(diff={diff[col]:.0f})")

    # Check 2: gender consistency per row
    if {_CT_TOTAL, _CT_MALE, _CT_FEMALE}.issubset(df.columns):
        gender_sum = df[_CT_MALE] + df[_CT_FEMALE]
        diff = (gender_sum - df[_CT_TOTAL]).abs()
        for age in diff.index:
            if not pd.isna(diff[age]) and diff[age] > tol:
                print(f"    [VALID gender {unit} {year}] age '{age}': "
                      f"m+f={gender_sum[age]:.0f} ≠ ogółem={df.at[age, _CT_TOTAL]:.0f} "
                      f"(diff={diff[age]:.0f})")

# ---------------------------------------------------------------------------
# Main cross-table building loop
# ---------------------------------------------------------------------------

cross_tables: dict[str, dict[str, pd.DataFrame]] = {}
# Structure: { name_pl (or 'Polska') → { '1986' → DataFrame } }

for year in YEARS:
    stem_total = f'pop_age_{year}_f'
    stem_men   = f'pop_age_men_{year}_f'

    print(f"\nYear {year}")

    df_total = _load_table(stem_total)
    df_men   = _load_table(stem_men)

    if df_total is None or df_men is None:
        print(f"  [SKIP year={year}] Could not load both tables")
        continue

    # Align to common canonical age columns
    age_cols_t = [c for c in _CANONICAL_AGE_COLS if c in df_total.columns]
    age_cols_m = [c for c in _CANONICAL_AGE_COLS if c in df_men.columns]
    age_cols   = [c for c in _CANONICAL_AGE_COLS if c in age_cols_t and c in age_cols_m]

    if not age_cols:
        print(f"  [SKIP year={year}] No common age columns")
        continue

    missing_in_men   = sorted(set(age_cols_t) - set(age_cols_m))
    missing_in_total = sorted(set(age_cols_m) - set(age_cols_t))
    if missing_in_men:
        print(f"  [WARN] Cols absent in men table:   {missing_in_men}")
    if missing_in_total:
        print(f"  [WARN] Cols absent in total table: {missing_in_total}")

    # Common territorial units
    units_total = set(df_total.index)
    units_men   = set(df_men.index)
    units_common = units_total & units_men
    only_total   = sorted(units_total - units_men)
    only_men     = sorted(units_men - units_total)
    if only_total:
        print(f"  [WARN] Units only in total table: {only_total}")
    if only_men:
        print(f"  [WARN] Units only in men table:   {only_men}")

    year_str = str(year)
    n_built  = 0

    for unit in sorted(units_common):
        row_total = df_total.loc[unit, age_cols]
        row_men   = df_men.loc[unit, age_cols]
        row_fem   = row_total - row_men

        df_cross = pd.DataFrame({
            _CT_TOTAL:  row_total.values,
            _CT_MALE:   row_men.values,
            _CT_FEMALE: row_fem.values,
        }, index=age_cols)
        df_cross.index.name = 'age_group'
        df_cross = df_cross.astype(float)

        _validate_cross_table(df_cross, unit, year)

        if unit not in cross_tables:
            cross_tables[unit] = {}
        cross_tables[unit][year_str] = df_cross
        n_built += 1

    print(f"  Built {n_built} cross-tables")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f"Territorial units in cross_tables: {len(cross_tables)}")
print(f"Units: {sorted(cross_tables.keys())}")
sample_unit = 'Warszawskie' if 'Warszawskie' in cross_tables else sorted(cross_tables.keys())[0]
sample_years = sorted(cross_tables[sample_unit].keys())
print(f"\nSample unit '{sample_unit}' — years: {sample_years}")
print(cross_tables[sample_unit][sample_years[0]])

# ---------------------------------------------------------------------------
# Save as pickle
# ---------------------------------------------------------------------------

pickle_path = gus_root / 'pop_age_cross_tables.pkl'
with open(pickle_path, 'wb') as fh:
    pickle.dump(cross_tables, fh)
print(f"\nSaved: {pickle_path}")



Year 1986
    Loaded from xlsx: pop_age_men_1986_f.xlsx
  Built 50 cross-tables

Year 1987
    Loaded from xlsx: pop_age_1987_f.xlsx
  Built 50 cross-tables

Year 1988
    Loaded from xlsx: pop_age_1988_f.xlsx
  Built 50 cross-tables

Year 1989
  Built 50 cross-tables

Year 1990
  Built 50 cross-tables

Year 1991
    Loaded from xlsx: pop_age_men_1991_f.xlsx
  Built 50 cross-tables

Year 1992
  Built 50 cross-tables

Year 1993
    Loaded from xlsx: pop_age_men_1993_f.xlsx
  Built 50 cross-tables

Year 1994
  Built 50 cross-tables

Territorial units in cross_tables: 50
Units: ['Bialskopodlaskie', 'Białostockie', 'Bielskie', 'Bydgoskie', 'Chełmskie', 'Ciechanowskie', 'Częstochowskie', 'Elbląskie', 'Gdańskie', 'Gorzowskie', 'Jeleniogórskie', 'Kaliskie', 'Katowickie', 'Kieleckie', 'Konińskie', 'Koszalińskie', 'Krakowskie', 'Krośnieńskie', 'Legnickie', 'Leszczyńskie', 'Lubelskie', 'Nowosądeckie', 'Olsztyńskie', 'Opolskie', 'Ostrołęckie', 'Pilskie', 'Piotrkowskie', 'Polska', 'Poznańskie', '

In [5]:

# =============================================================================
# GUS02D PART 3: Extract & validate sex × education tables
#                from pop__tot_age_men_educ_YYYY [meta].json
# =============================================================================
#
# Structure of source files:
#   All files: one element (table or paragraph), units in THOUSANDS.
#   Rows: multi-year time series, three sex groups:
#       Ogółem / Mężczyźni / Kobiety  (each with several historical years)
#   Columns: ogółem | wyższe | średnie | zasadnicze zawodowe | podstawowe
#
#   We extract the ROW corresponding to the LATEST year of each sex group,
#   then build a 3-row DataFrame  (sex × education)  in thousands.
#
#   1986: no <table> element – data is in paragraph text → parse manually.
#
# Files needing review → saved as pop__tot_age_men_educ_YYYY_validate.xlsx
# =============================================================================

import json, re, unicodedata, pickle
from io import StringIO
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Load raw JSON files
# ---------------------------------------------------------------------------
EDUC_YEARS         = [1986, 1987, 1991, 1992, 1993, 1994]
EDUC_SPECIAL_YEAR  = 1988

educ_jsons: dict[str, str] = {}
for year in EDUC_YEARS:
    fname = f"pop__tot_age_men_educ_{year} [meta].json"
    path  = json_root / fname
    if path.exists():
        educ_jsons[str(year)] = path.read_text()
        print(f"Loaded: {fname}")
    else:
        print(f"[MISSING] {fname}")

educ_special_json: str | None = None
special_path = json_root / f"pop__tot_age_men_educ_{EDUC_SPECIAL_YEAR} [meta].json"
if special_path.exists():
    educ_special_json = special_path.read_text()
    print(f"Loaded: {special_path.name}")
else:
    print(f"[MISSING] {special_path.name}")

# ---------------------------------------------------------------------------
# Canonical column names for sex × education table
# ---------------------------------------------------------------------------
_EDUC_SEX_ROWS  = ['ogółem', 'mężczyźni', 'kobiety']
_EDUC_COLS      = ['ogółem', 'wyższe', 'średnie', 'zasadnicze zawodowe', 'podstawowe']
# Values are in thousands in the regular tables

# OCR → canonical education column mapping (positional: col 0=ogółem, 1=wyższe, …)
_EDUC_COL_POS   = _EDUC_COLS   # length 5; position is reliable since OCR headers are messy

# OCR patterns to detect which sex-group block a row starts
_SEX_PATTERNS = [
    (re.compile(r'\bmezczyzni\b|\bmales?\b|\bmezczyz\b', re.I), 'mężczyźni'),
    (re.compile(r'\bkobiety\b|\bfemales?\b|\bkobiet\b',  re.I), 'kobiety'),
    (re.compile(r'\boglem\b|\bog[oó]lem\b|\btotal\b',    re.I), 'ogółem'),
]

def _detect_sex(cell_str: str) -> str | None:
    s = str(cell_str)
    for pat, label in _SEX_PATTERNS:
        if pat.search(s):
            return label
    return None

def _parse_year_from_cell(cell_str: str) -> int | None:
    """Extract a 4-digit year (1960–2010) from an OCR cell string."""
    m = re.search(r'\b(19[6-9]\d|20[01]\d)\b', str(cell_str))
    return int(m.group(1)) if m else None

def _clean_thousands(val) -> float:
    """Parse a value that is in thousands; strip OCR noise."""
    if isinstance(val, (int, float)):
        return float(val) if not pd.isna(val) else float('nan')
    s = re.sub(r'[^\d]', '', str(val))
    return float(s) if s else float('nan')

# ---------------------------------------------------------------------------
# Parse paragraph text (1986 only)
# ---------------------------------------------------------------------------
def _parse_paragraph_1986(text: str) -> pd.DataFrame | None:
    """
    Parse the raw paragraph text of 1986 into a 3×5 DataFrame (sex × edu).
    The text contains lines like:
      'OGLEM 1960 20004d 415 2046 630 7838'
      '1986 27929 1642 6597 5765 11822'
      'Mezczyzni 1960 9260 ...'
      '1986 13387 878 2687 3749 5364'
      'Kobiety 1960 10744 ...'
      '1986 14542 764 3910 2016 6458'
    We want the row for 1986 in each sex block.
    """
    # Normalize: replace multiple spaces/newlines with single space
    text = re.sub(r'[\r\n]+', '\n', text)
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    result: dict[str, dict[str, float]] = {}
    current_sex: str | None = None

    for line in lines:
        tokens = line.split()
        if not tokens:
            continue

        # Detect sex-group header in first token(s)
        header_text = ' '.join(tokens[:3])
        sex = _detect_sex(header_text)
        if sex:
            current_sex = sex

        # Try to find a year + 5 numeric values anywhere in this line
        # e.g. '1986 27929 1642 6597 5765 11822'
        #  or  'OGLEM 1986 27929 1642 6597 5765 11822'
        numbers = [t for t in tokens if re.match(r'^\d{4,}[a-z]?$', t, re.I)]
        # Find the year token
        year_match = None
        for tok in tokens:
            y = _parse_year_from_cell(tok)
            if y:
                year_match = y
                break
        if year_match != 1986 or current_sex is None:
            continue

        # Collect values after the year token
        pos = next((i for i, t in enumerate(tokens) if _parse_year_from_cell(t) == 1986), None)
        if pos is None:
            continue
        val_tokens = tokens[pos + 1:]
        vals = []
        for t in val_tokens:
            clean = re.sub(r'[^\d]', '', t)
            if clean:
                vals.append(float(clean))
        if len(vals) >= 5:
            vals = vals[:5]
            result[current_sex] = dict(zip(_EDUC_COLS, vals))

    if not result:
        return None

    df = pd.DataFrame(result).T
    df = df[[c for c in _EDUC_COLS if c in df.columns]]
    df.index.name = 'sex'
    return df.astype(float)

# ---------------------------------------------------------------------------
# Parse regular table-based files (1987, 1991–1994)
# ---------------------------------------------------------------------------
def _parse_educ_table(raw_json: str, year: int) -> pd.DataFrame | None:
    """
    Parse a regular edu×sex table from JSON. Returns DataFrame (sex × education)
    with the row for `year`, units in thousands.
    """
    data   = json.loads(raw_json)
    elems  = data['elements']

    # --- Try table element first ---
    tables = [e for e in elems if e['category'] == 'table']
    if tables:
        html = tables[0]['content']['html']
        raw  = pd.read_html(StringIO(html))[0]

        # Flatten MultiIndex columns → just use positional slots
        # Col 0 = WYSZCZEGLNIENIE, col 1 = ogółem, cols 2–5 = edu levels
        raw.columns = range(len(raw.columns))

        current_sex: str | None = None
        result: dict[str, dict[str, float]] = {}

        for _, row in raw.iterrows():
            cell0 = str(row[0])
            sex = _detect_sex(cell0)
            if sex:
                current_sex = sex

            yr = _parse_year_from_cell(cell0)
            if yr != year or current_sex is None:
                continue

            vals = [_clean_thousands(row[c]) for c in range(1, min(6, len(row)))]
            if len(vals) >= 5:
                result[current_sex] = dict(zip(_EDUC_COLS, vals[:5]))

        if not result:
            return None

        df = pd.DataFrame(result).T
        df = df[[c for c in _EDUC_COLS if c in df.columns]]
        df.index.name = 'sex'
        return df.astype(float)

    # --- Fallback: paragraph text ---
    paras = [e for e in elems if e['category'] == 'paragraph']
    if paras:
        text = paras[0]['content'].get('text', '')
        return _parse_paragraph_1986(text)

    return None

# ---------------------------------------------------------------------------
# Validation: rows must sum correctly
# ---------------------------------------------------------------------------
def _validate_educ(df: pd.DataFrame, label: str, tol: float = 0.6) -> list[str]:
    """
    In thousands: mężczyźni + kobiety ≈ ogółem  (tol = 0.5 for rounding).
    Also: wyższe + średnie + zasadnicze + podstawowe ≤ ogółem (some unschooled/unknown).
    """
    warns = []
    if {'ogółem', 'mężczyźni', 'kobiety'}.issubset(df.index):
        for col in df.columns:
            s = df.at['mężczyźni', col] + df.at['kobiety', col]
            t = df.at['ogółem', col]
            if not (pd.isna(s) or pd.isna(t)) and abs(s - t) > tol:
                warns.append(
                    f"  col '{col}': mężczyźni+kobiety={s:.1f} ≠ ogółem={t:.1f} "
                    f"(diff={abs(s-t):.1f})")
    if 'ogółem' in df.index:
        for row_sex in df.index:
            row = df.loc[row_sex]
            if 'ogółem' in df.columns:
                edu_cols = [c for c in _EDUC_COLS[1:] if c in df.columns]
                edu_sum  = row[edu_cols].sum()
                total    = row['ogółem']
                if not (pd.isna(edu_sum) or pd.isna(total)) and edu_sum > total + tol:
                    warns.append(
                        f"  sex '{row_sex}': Σedu={edu_sum:.1f} > ogółem={total:.1f} "
                        f"(diff={edu_sum - total:.1f})")
    return warns

# ---------------------------------------------------------------------------
# Main extraction loop
# ---------------------------------------------------------------------------
save_dir_educ = gus_root / 'data' / 'extracted_tables'
save_dir_educ.mkdir(parents=True, exist_ok=True)

educ_tables: dict[str, pd.DataFrame] = {}   # year_str → df  (sex × education)
educ_issues: dict[str, list[str]]    = {}

all_to_process = {**{str(y): educ_jsons[str(y)] for y in EDUC_YEARS if str(y) in educ_jsons}}

for year_str, raw_json in sorted(all_to_process.items()):
    yr = int(year_str)
    label = f"pop__tot_age_men_educ_{yr}"
    print(f"\n{label}")

    df = _parse_educ_table(raw_json, yr)

    if df is None:
        print("  [ERROR] Extraction failed → saved raw for review")
        # Save empty placeholder xlsx
        xlsx_path = save_dir_educ / f"{label}_validate.xlsx"
        pd.DataFrame().to_excel(xlsx_path)
        educ_issues[year_str] = ["extraction failed"]
        continue

    educ_tables[year_str] = df

    # Coverage check
    missing_rows = [s for s in _EDUC_SEX_ROWS if s not in df.index]
    missing_cols = [c for c in _EDUC_COLS    if c not in df.columns]
    if missing_rows:
        print(f"  Missing sex rows: {missing_rows}")
    if missing_cols:
        print(f"  Missing edu cols: {missing_cols}")

    # Validate sums
    warns = _validate_educ(df, label)
    if warns:
        educ_issues[year_str] = warns
        print(f"  ✗ Validation issues ({len(warns)}):")
        for w in warns:
            print(w)

    needs_review = bool(warns) or bool(missing_rows) or bool(missing_cols)
    if needs_review:
        if year_str not in educ_issues:
            educ_issues[year_str] = []
        xlsx_path = save_dir_educ / f"{label}_validate.xlsx"
        df.to_excel(xlsx_path)
        reasons = []
        if warns:
            reasons.append(f"{len(warns)} sum mismatch(es)")
        if missing_rows:
            reasons.append(f"missing rows: {missing_rows}")
        if missing_cols:
            reasons.append(f"missing cols: {missing_cols}")
        print(f"  → Saved for review ({', '.join(reasons)}): {xlsx_path.name}")
    else:
        print(f"  ✓  {df.shape[0]} sex groups × {df.shape[1]} edu cols — OK")
        print(df.to_string())

print(f"\n{'='*60}")
print(f"Extracted:    {len(educ_tables)}/{len(all_to_process)} regular files")
print(f"Need review:  {len(educ_issues)} file(s): {sorted(educ_issues.keys())}")
print(f"Review dir:   {save_dir_educ}")


Loaded: pop__tot_age_men_educ_1986 [meta].json
Loaded: pop__tot_age_men_educ_1987 [meta].json
Loaded: pop__tot_age_men_educ_1991 [meta].json
Loaded: pop__tot_age_men_educ_1992 [meta].json
Loaded: pop__tot_age_men_educ_1993 [meta].json
Loaded: pop__tot_age_men_educ_1994 [meta].json
Loaded: pop__tot_age_men_educ_1988 [meta].json

pop__tot_age_men_educ_1986
  ✓  3 sex groups × 5 edu cols — OK
            ogółem  wyższe  średnie  zasadnicze zawodowe  podstawowe
sex                                                                 
ogółem     27929.0  1642.0   6597.0               5765.0     11822.0
mężczyźni  13387.0   878.0   2687.0               3749.0      5364.0
kobiety    14542.0   764.0   3910.0               2016.0      6458.0

pop__tot_age_men_educ_1987
  ✓  3 sex groups × 5 edu cols — OK
            ogółem  wyższe  średnie  zasadnicze zawodowe  podstawowe
sex                                                                 
ogółem     28101.0  1679.0   6691.0               5919.0    

In [6]:

# =============================================================================
# GUS02D PART 4: Parse special 1988 file (education × age) and build
#               all sex×education cross-tables + save pickle
# =============================================================================
#
# 1988 special table structure (absolute numbers, NOT thousands):
#   Rows: education levels × sex  (0=ogółem, m=mężczyźni, k=kobiety)
#         education groups: Wyższe, Średnie, Zasadnicze zawodowe,
#                           Podstawowe, Niepełne podstawowe i bez wykształcenia
#   Columns: ogółem | W tym czynni zawodowo | 15-17 | 18-24 | 25-29 |
#             30-39 | 40-49 | 50-59 | 60 lat i więcej
#
# We produce TWO outputs from 1988:
#   A) educ_tables['1988']         — sex × education  (ogółem column only, /1000 → thousands)
#   B) educ_age_table_1988         — education × age  (3 sheets: ogółem/mężczyźni/kobiety)
#
# For ALL years: build cross-table  sex × education  and store in dict.
# =============================================================================

import json, re, pickle
from io import StringIO
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Canonical labels for 1988 special
# ---------------------------------------------------------------------------
_EDUC_LABELS_1988 = [
    'wyższe',
    'średnie',
    'zasadnicze zawodowe',
    'podstawowe',
    'niepełne podstawowe i bez wykształcenia',
]
_AGE_COLS_1988 = [
    'ogółem', 'w tym czynni zawodowo',
    '15-17', '18-24', '25-29', '30-39', '40-49', '50-59', '60 i więcej',
]

def _norm_str(s: str) -> str:
    s = str(s).lower().strip()
    for f, t in [('ł','l'),('ą','a'),('ę','e'),('ó','o'),('ś','s'),
                 ('ź','z'),('ż','z'),('ć','c'),('ń','n'),('0','o')]:
        s = s.replace(f, t)
    s = re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', s))
    return s

_EDUC_PATTERNS_1988 = [
    (re.compile(r'wyzsze|higher',          re.I), 'wyższe'),
    (re.compile(r'srednie|secondary',      re.I), 'średnie'),
    (re.compile(r'zasadnicze|zawodowe|basic voca', re.I), 'zasadnicze zawodowe'),
    (re.compile(r'podstawo(?!we\s+i)|primary', re.I), 'podstawowe'),
    (re.compile(r'niepe|bez wyksztal|bez szk|without', re.I),
     'niepełne podstawowe i bez wykształcenia'),
]

def _detect_educ(cell_str: str) -> str | None:
    for pat, label in _EDUC_PATTERNS_1988:
        if pat.search(str(cell_str)):
            return label
    return None

def _detect_sex_short(cell_str: str) -> str | None:
    """Detect sex from single-character 'm'/'k' or longer strings."""
    s = str(cell_str).strip()
    if s == 'm':
        return 'mężczyźni'
    if s == 'k':
        return 'kobiety'
    if re.search(r'\bmezczyzni\b|\bmales\b', s, re.I):
        return 'mężczyźni'
    if re.search(r'\bkobiety\b|\bfemales\b', s, re.I):
        return 'kobiety'
    if re.search(r'oglem|ogolem|^0$|^\s*0\s+glem', s, re.I):
        return 'ogółem'
    return None

def _parse_two_in_one_cell(cell_str: str) -> tuple[float, float] | None:
    """
    Some cells contain two values merged: '28268775 13553887'.
    Returns (total_val, men_val) or None.
    """
    parts = re.findall(r'\d+', str(cell_str))
    if len(parts) >= 2:
        return float(parts[0]), float(parts[1])
    return None

def _parse_special_1988(raw_json: str) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]] | tuple[None, None]:
    """
    Returns:
      (sex_educ_df, age_dfs)
      sex_educ_df  : DataFrame  rows=education, cols=['ogółem','mężczyźni','kobiety']  (in thousands)
      age_dfs      : dict { 'ogółem'/'mężczyźni'/'kobiety' → DataFrame(education × age) } (absolute)
    """
    data   = json.loads(raw_json)
    tables = [e for e in data['elements'] if e['category'] == 'table']
    if not tables:
        print("  [ERROR 1988] No table element found")
        return None, None

    raw = pd.read_html(StringIO(tables[0]['content']['html']))[0]
    raw.columns = range(len(raw.columns))

    # State machine: track current education group and sex
    current_educ: str | None    = None
    # Storage: {educ: {sex: {age_col: value}}}
    store: dict[str, dict[str, dict[str, float]]] = {}
    # Also track the OGLEM totals row (first row of each education block)
    oglem_store: dict[str, dict[str, float]] = {}  # educ → {age_col: val}

    def _store_age_row(educ: str, sex: str, vals: list[float]):
        if educ not in store:
            store[educ] = {}
        if sex not in store[educ]:
            store[educ][sex] = {}
        for col, v in zip(_AGE_COLS_1988, vals):
            store[educ][sex][col] = v

    for _, row in raw.iterrows():
        cell0 = str(row[0]).strip()

        # Skip header / unit rows
        if re.search(r'liczbach|bezwzglednych|tysiacach|wyksztalcenie:', cell0, re.I):
            if re.search(r'wyksztalcenie:', cell0, re.I):
                current_educ = None
            continue

        # Detect section headers: 0 glem / Ogółem block
        if re.search(r'^0\s*glem$|^oglem\s*Total$', cell0, re.I):
            current_educ = None
            continue

        # Detect education group start (and possibly ogółem sex within it)
        educ = _detect_educ(cell0)
        if educ:
            current_educ = educ
            # Does this cell also contain the ogółem-sex numeric row?
            # e.g. 'Wyzsze Q' → values follow; or 'Zasadnicze zawodowe 0'
            # Check if columns 1..8 have valid numbers
            vals_raw = [row[c] for c in range(1, min(10, len(row)))]
            # First value might be two-merged (ogółem+mężczyźni in one cell)
            first = str(row[1]) if len(row) > 1 else ''
            merged = _parse_two_in_one_cell(first)
            if merged:
                # This row contains 'OGLEM m' combined in cell1
                # ogółem values from cell1 (first of pair) + cells 2..8
                oglem_vals = [merged[0]] + [float(re.sub(r'[^\d]','',str(row[c])) or 'nan')
                                             for c in range(2, min(10, len(row)))]
                men_vals   = [merged[1]] + oglem_vals[1:]  # reuse age vals — they are shared
                # Actually merged cells have separate values for each sex in different cells
                # We'll handle below in the 'OGLEM m' row pattern
                pass
            # Try to parse ogółem sex row from this education row
            sex = _detect_sex_short(cell0)
            if not sex:
                # cell0 might be 'Wyzsze Q' without sex → ogółem of this educ group comes separately
                # Try reading numeric values for ogółem
                vals = []
                for c in range(1, min(10, len(row))):
                    cleaned = re.sub(r'[^\d]', '', str(row[c]))
                    vals.append(float(cleaned) if cleaned else float('nan'))
                if any(not pd.isna(v) for v in vals):
                    _store_age_row(educ, 'ogółem', vals[:len(_AGE_COLS_1988)])
            elif sex == 'ogółem':
                vals = []
                for c in range(1, min(10, len(row))):
                    cleaned = re.sub(r'[^\d]', '', str(row[c]))
                    vals.append(float(cleaned) if cleaned else float('nan'))
                _store_age_row(educ, 'ogółem', vals[:len(_AGE_COLS_1988)])
            continue

        # Detect combined OGLEM+m row: 'OGLEM 。 m' with two-merged cell1
        if re.search(r'oglem.*\bm\b', cell0, re.I) or re.search(r'0\s*glem.*\bm\b', cell0, re.I):
            current_educ = None  # top-level total, before any education group
            first_cell = str(row[1]) if len(row) > 1 else ''
            merged = _parse_two_in_one_cell(first_cell)
            if merged:
                # Build ogółem row
                oglem_vals = [merged[0]]
                for c in range(2, min(10, len(row))):
                    val_raw = str(row[c])
                    # Some cells also have two numbers merged for sub-age groups
                    sub = _parse_two_in_one_cell(val_raw)
                    oglem_vals.append(float(re.sub(r'[^\d]','',val_raw.split()[0]) or 'nan')
                                      if not sub else sub[0])
                men_vals = [merged[1]]
                for c in range(2, min(10, len(row))):
                    val_raw = str(row[c])
                    sub = _parse_two_in_one_cell(val_raw)
                    men_vals.append(sub[1] if sub else float('nan'))
                _store_age_row('__total__', 'ogółem',    oglem_vals[:len(_AGE_COLS_1988)])
                _store_age_row('__total__', 'mężczyźni', men_vals[:len(_AGE_COLS_1988)])
            continue

        # Single-character sex continuation row (m/k) under currently active education group
        sex = _detect_sex_short(cell0)
        if sex and current_educ:
            vals = []
            for c in range(1, min(10, len(row))):
                cleaned = re.sub(r'[^\d]', '', str(row[c]))
                vals.append(float(cleaned) if cleaned else float('nan'))
            _store_age_row(current_educ, sex, vals[:len(_AGE_COLS_1988)])
            continue

        # kobiety follow-up row for top-level (cell0 == 'k')
        if cell0 == 'k' and current_educ is None:
            vals = []
            for c in range(1, min(10, len(row))):
                cleaned = re.sub(r'[^\d]', '', str(row[c]))
                vals.append(float(cleaned) if cleaned else float('nan'))
            _store_age_row('__total__', 'kobiety', vals[:len(_AGE_COLS_1988)])

    # --- Build age DataFrames per sex: education × age columns (absolute) ---
    age_dfs: dict[str, pd.DataFrame] = {}
    all_sex = ['ogółem', 'mężczyźni', 'kobiety']
    educ_order = _EDUC_LABELS_1988

    for sex in all_sex:
        rows = {}
        for educ in educ_order:
            if educ in store and sex in store[educ]:
                rows[educ] = store[educ][sex]
        if rows:
            df_age = pd.DataFrame(rows).T
            df_age = df_age[[c for c in _AGE_COLS_1988 if c in df_age.columns]]
            df_age.index.name = 'education'
            age_dfs[sex] = df_age.astype(float)

    # --- Build sex × education table (ogółem column only, in thousands) ---
    # Take the 'ogółem' age column (= total for each education group)
    sex_educ_rows: dict[str, dict[str, float]] = {}
    for sex in all_sex:
        row_vals: dict[str, float] = {}
        for educ in ['ogółem'] + educ_order:  # 'ogółem' = total education (from __total__)
            if educ == 'ogółem':
                # total = __total__ store
                src = store.get('__total__', {}).get(sex, {})
                row_vals['ogółem'] = src.get('ogółem', float('nan')) / 1000.0
            else:
                src = store.get(educ, {}).get(sex, {})
                row_vals[educ] = src.get('ogółem', float('nan')) / 1000.0
        sex_educ_rows[sex] = row_vals

    sex_educ_cols = ['ogółem'] + educ_order
    sex_educ_df   = pd.DataFrame(sex_educ_rows).T
    sex_educ_df   = sex_educ_df[[c for c in sex_educ_cols if c in sex_educ_df.columns]]
    sex_educ_df.index.name = 'sex'

    return sex_educ_df, age_dfs

# ---------------------------------------------------------------------------
# Parse and validate 1988
# ---------------------------------------------------------------------------
print("Processing 1988 special file …")
educ_1988_sex_educ, educ_1988_age = _parse_special_1988(educ_special_json)

if educ_1988_sex_educ is not None:
    educ_tables['1988'] = educ_1988_sex_educ
    print("\n  sex × education (in thousands):")
    print(educ_1988_sex_educ.to_string())
    warns_1988 = _validate_educ(educ_1988_sex_educ, '1988')
    needs_review_1988 = bool(warns_1988) or any(
        s not in educ_1988_sex_educ.index for s in _EDUC_SEX_ROWS)
    if warns_1988:
        print(f"  ✗ Validation issues:")
        for w in warns_1988:
            print(w)
    else:
        print("  ✓ sex×education sums OK")
    if educ_1988_age:
        print("\n  education × age sub-tables:")
        for sex, df_a in educ_1988_age.items():
            print(f"    [{sex}]  shape={df_a.shape}")
            print(df_a.to_string())
    if needs_review_1988:
        xlsx_1988 = save_dir_educ / "pop__tot_age_men_educ_1988_validate.xlsx"
        with pd.ExcelWriter(xlsx_1988) as writer:
            educ_1988_sex_educ.to_excel(writer, sheet_name='sex_education')
            if educ_1988_age:
                for sex, df_a in educ_1988_age.items():
                    sheet = sex[:10].replace('ę','e').replace('ó','o').replace('ź','z')
                    df_a.to_excel(writer, sheet_name=f'age_{sheet}')
        print(f"\n  → Saved for review: {xlsx_1988.name}")
    else:
        # Save clean version for Part 5 loader
        xlsx_1988_clean = save_dir_educ / "pop__tot_age_men_educ_1988.xlsx"
        with pd.ExcelWriter(xlsx_1988_clean) as writer:
            educ_1988_sex_educ.to_excel(writer, sheet_name='sex_education')
            if educ_1988_age:
                for sex, df_a in educ_1988_age.items():
                    sheet = sex[:10].replace('ę','e').replace('ó','o').replace('ź','z')
                    df_a.to_excel(writer, sheet_name=f'age_{sheet}')
        print(f"\n  → Saved clean: {xlsx_1988_clean.name}")
else:
    print("  [ERROR] 1988 extraction completely failed")

# ---------------------------------------------------------------------------
# Build final sex × education cross-tables dict and save pickle
# ---------------------------------------------------------------------------
# Structure: educ_cross_tables[year_str] = {
#     'sex_education':  DataFrame(sex × education)  — units: thousands
#     'education_age':  dict(sex → DataFrame(education × age))  — 1988 only, absolute
# }
#
# For years where manual xlsx correction exists: load from xlsx.
# ---------------------------------------------------------------------------

def _load_educ_table(year_str: str) -> pd.DataFrame | None:
    """Load sex × education table: corrected xlsx → extracted → re-parse."""
    stem = f"pop__tot_age_men_educ_{year_str}"
    xlsx_clean = save_dir_educ / f"{stem}.xlsx"
    if xlsx_clean.exists():
        df = pd.read_excel(xlsx_clean, sheet_name='sex_education' if year_str == '1988' else 0,
                           index_col=0)
        df.columns = [str(c) for c in df.columns]
        print(f"    Loaded xlsx: {xlsx_clean.name}")
        return df.astype(float)
    if year_str in educ_tables:
        return educ_tables[year_str]
    return None

def _load_educ_age_1988() -> dict[str, pd.DataFrame] | None:
    """Load the education × age tables for 1988."""
    stem = "pop__tot_age_men_educ_1988"
    xlsx_clean    = save_dir_educ / f"{stem}.xlsx"
    xlsx_validate = save_dir_educ / f"{stem}_validate.xlsx"
    source = xlsx_clean if xlsx_clean.exists() else (xlsx_validate if xlsx_validate.exists() else None)
    if source:
        xl    = pd.ExcelFile(source)
        dfs   = {}
        sx_map = {'age_ogolem': 'ogółem', 'age_mezczyzni': 'mężczyźni', 'age_kobiety': 'kobiety',
                  'age_ogłem': 'ogółem', 'age_mezczyni': 'mężczyźni'}  # OCR fallbacks
        for sheet in xl.sheet_names:
            if sheet.startswith('age_'):
                df = pd.read_excel(source, sheet_name=sheet, index_col=0).astype(float)
                df.columns = [str(c) for c in df.columns]
                sex_label = sx_map.get(sheet, sheet.replace('age_',''))
                dfs[sex_label] = df
        if dfs:
            return dfs
    # Fallback: use in-memory
    return educ_1988_age if educ_1988_age else None

ALL_EDUC_YEARS = sorted([str(y) for y in EDUC_YEARS] + ['1988'])

educ_cross_tables: dict[str, dict] = {}

print(f"\n{'='*60}")
print("Building education cross-tables …")

for year_str in ALL_EDUC_YEARS:
    df_sex_educ = _load_educ_table(year_str)
    if df_sex_educ is None:
        print(f"  {year_str}: [SKIP] could not load sex×education table")
        continue

    entry: dict = {'sex_education': df_sex_educ}

    if year_str == '1988':
        age_dfs = _load_educ_age_1988()
        if age_dfs:
            entry['education_age'] = age_dfs

    # Re-validate
    warns = _validate_educ(df_sex_educ, year_str)
    status = "✓" if not warns else "✗"
    print(f"  {year_str}: {status}  sex×edu shape={df_sex_educ.shape}"
          + (f"  age_sheets={list(entry.get('education_age',{}).keys())}"
             if 'education_age' in entry else ''))
    for w in warns:
        print(w)

    educ_cross_tables[year_str] = entry

# Save pickle
pickle_path_educ = gus_root / 'pop_educ_cross_tables.pkl'
with open(pickle_path_educ, 'wb') as fh:
    pickle.dump(educ_cross_tables, fh)

print(f"\nSaved {len(educ_cross_tables)} year entries → {pickle_path_educ}")
print("Years:", sorted(educ_cross_tables.keys()))


Processing 1988 special file …

  sex × education (in thousands):
              ogółem    wyższe   średnie  zasadnicze zawodowe  podstawowe  niepełne podstawowe i bez wykształcenia
sex                                                                                                               
ogółem     28268.775  1838.331  6979.573             6665.767    1721.149                                  613.304
mężczyźni  13553.887   975.000  2793.653             4274.207    4859.277                                      NaN
kobiety    14714.888   863.331  4185.920             2391.560    6102.116                                      NaN
  ✗ Validation issues:
  col 'podstawowe': mężczyźni+kobiety=10961.4 ≠ ogółem=1721.1 (diff=9240.2)

  education × age sub-tables:
    [ogółem]  shape=(5, 9)
                                            ogółem  w tym czynni zawodowo    15-17      18-24      25-29      30-39      40-49     50-59  60 i więcej
education                                           